In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import optuna
import joblib

In [3]:
df_maths = pd.read_csv(r"..\data\student-mat.csv", delimiter=";")
df_por = pd.read_csv(r"..\data\student-por.csv", delimiter=";")

df_maths['subject'] = 'Maths'
df_por['subject']= 'Portuguese'

df = pd.concat([df_maths, df_por], axis=0).reset_index(drop=True)

id_cols = ["school", "sex", "age", "address", "famsize", "Pstatus", 
           "Medu", "Fedu", "Mjob", "Fjob", "reason", "nursery", "internet"]

df['student_id'] = df[id_cols].astype(str).agg('-'.join, axis=1)

#  target
target = 'G3'

# feature sets
demographic_features = [col for col in df.columns if col not in ['G1', 'G2', 'G3', 'student_id']]

# Categorical columns 
cat_features = df[demographic_features].select_dtypes(include=['str']).columns.tolist()
for col in cat_features:
    df[col] = df[col].astype('category') 

# feature sets for  3 models
features_model1 = demographic_features
features_model2 = demographic_features + ['G1']
features_model3 = demographic_features + ['G1', 'G2']

## Model 2 (After G1)

In [4]:
X = df[features_model2]
y = df['G3']
groups = df['student_id']

def objective(trial):

    params = {"tree_method": "hist","objective": "reg:squarederror","random_state": 42,
              "enable_categorical" :True, "n_estimators":1000,  #"early_stopping_rounds":50,
              
              "n_estimators": trial.suggest_int("n_estimators",200,1000),
              "max_depth": trial.suggest_int("max_depth",3,10),
              "learning_rate": trial.suggest_float("learning_rate",0.005,0.1,log=True),
              "subsample": trial.suggest_float("subsample",0.5,1.0),
              "colsample_bytree": trial.suggest_float("colsample_bytree",0.5,1.0),
              "min_child_weight": trial.suggest_int("min_child_weight",1,15),
              "gamma": trial.suggest_float("gamma",0,5),
              #"reg_alpha": trial.suggest_float("reg_alpha",1e-4,10,log=True),
              "reg_lambda": trial.suggest_float("reg_lambda",1e-4,10,log=True),
    }
    
    gkf = GroupKFold(n_splits=5)
    mae_scores = []

    for train_idx, val_idx in gkf.split(X, y, groups=groups):

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = xgb.XGBRegressor(**params)
        
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        preds = model.predict(X_val)
        mae_scores.append(mean_absolute_error(y_val, preds))

    return np.mean(mae_scores)

In [5]:
study = optuna.create_study(
    direction="minimize"
)

study.optimize(
    objective,
    n_trials=200,
    show_progress_bar=True
)

[I 2026-05-13 04:05:21,609] A new study created in memory with name: no-name-fe1c3d08-3ba0-4178-ab3c-818874e9d82c


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-05-13 04:05:23,780] Trial 0 finished with value: 1.3540875911712646 and parameters: {'n_estimators': 662, 'max_depth': 10, 'learning_rate': 0.03302731336021678, 'subsample': 0.5465369385301492, 'colsample_bytree': 0.9906332163787257, 'min_child_weight': 9, 'gamma': 2.8868211091090643, 'reg_lambda': 0.0032612065058309227}. Best is trial 0 with value: 1.3540875911712646.
[I 2026-05-13 04:05:26,567] Trial 1 finished with value: 1.4301870346069336 and parameters: {'n_estimators': 699, 'max_depth': 8, 'learning_rate': 0.05919434844053432, 'subsample': 0.7259078522210154, 'colsample_bytree': 0.9885616768539893, 'min_child_weight': 13, 'gamma': 0.0896759254817392, 'reg_lambda': 0.003899948770721614}. Best is trial 0 with value: 1.3540875911712646.
[I 2026-05-13 04:05:27,931] Trial 2 finished with value: 1.4349199056625366 and parameters: {'n_estimators': 628, 'max_depth': 6, 'learning_rate': 0.0913674979267306, 'subsample': 0.965440899011079, 'colsample_bytree': 0.5456997965773114, 'm

In [6]:
trial_df = study.trials_dataframe()
fig = px.line(
    trial_df,
    x="number",
    y="value",
    title="Optuna Optimization History",template = "presentation"
)
fig.show()

In [7]:
print("Best MAE:")
print(study.best_value)

print("\nBest Parameters:")
print(study.best_params)

Best MAE:
1.3183340311050415

Best Parameters:
{'n_estimators': 422, 'max_depth': 4, 'learning_rate': 0.044612033222283375, 'subsample': 0.9604275190847189, 'colsample_bytree': 0.9792930197363612, 'min_child_weight': 3, 'gamma': 4.385668613911848, 'reg_lambda': 3.569836871987955}


In [8]:
def evaluate(df, feature_list, best_params, target_col='G3'):
    X = df[feature_list]
    y = df[target_col]
    groups = df['student_id']
    
    gkf = GroupKFold(n_splits=5)
    
    mae_scores = []
    mse_scores = []
    r2_scores = []
    
    print(f"\n--- Training Model with {len(feature_list)} features ---")
    
    for train_idx, val_idx in gkf.split(X, y, groups=groups):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = xgb.XGBRegressor(**best_params, tree_method= "hist",objective= "reg:squarederror",random_state= 42,
              enable_categorical =True)
        
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        preds = model.predict(X_val)
        mae_scores.append(mean_absolute_error(y_val, preds))
        mse_scores.append(mean_squared_error(y_val, preds))
        r2_scores.append(r2_score(y_val, preds))
        
    print('MAE :',mae_scores)    
    print(f"Average MAE: {np.mean(mae_scores):.4f}")
    print('MSE :',mse_scores)    
    print(f"Average MSE: {np.mean(mse_scores):.4f}")
    print('r2_scores : ',r2_scores)
    print(f"Average R2: {np.mean(r2_scores):.4f}")

In [9]:
# Evaluating Model 2 (After G1)
evaluate(df, features_model2, best_params=study.best_params)


--- Training Model with 32 features ---
MAE : [1.273879051208496, 1.3527376651763916, 1.333192229270935, 1.3188145160675049, 1.3130466938018799]
Average MAE: 1.3183
MSE : [3.4704980850219727, 3.5349392890930176, 3.53712797164917, 4.1342387199401855, 4.051673889160156]
Average MSE: 3.7457
r2_scores :  [0.7466739416122437, 0.7169796228408813, 0.7639843225479126, 0.7430263757705688, 0.7635051608085632]
Average R2: 0.7468


In [10]:
# final model 2 (using the whole data)
model_2 = xgb.XGBRegressor(**study.best_params, tree_method= "hist",objective= "reg:squarederror",random_state= 42,
              enable_categorical =True)
model_2.fit(
            X, y,
            verbose=True
        )

pred = model_2.predict(X)
mean_absolute_error(pred,y)

1.0099622011184692

In [11]:
import json

# Save the model itself
model_2.save_model("model_2.json")

## Model 3 (Final prediction after G2)

In [12]:
X = df[features_model3]
y = df['G3']
groups = df['student_id']

def objective(trial):

    params = {"tree_method": "hist","objective": "reg:squarederror","random_state": 42,
              "enable_categorical" :True, "n_estimators":1000,  #"early_stopping_rounds":50,
              
              "n_estimators": trial.suggest_int("n_estimators",200,1000),
              "max_depth": trial.suggest_int("max_depth",3,10),
              "learning_rate": trial.suggest_float("learning_rate",0.005,0.1,log=True),
              "subsample": trial.suggest_float("subsample",0.5,1.0),
              "colsample_bytree": trial.suggest_float("colsample_bytree",0.5,1.0),
              "min_child_weight": trial.suggest_int("min_child_weight",1,15),
              "gamma": trial.suggest_float("gamma",0,5),
              #"reg_alpha": trial.suggest_float("reg_alpha",1e-4,10,log=True),
              "reg_lambda": trial.suggest_float("reg_lambda",1e-4,10,log=True),
    }
    
    gkf = GroupKFold(n_splits=5)
    mae_scores = []

    for train_idx, val_idx in gkf.split(X, y, groups=groups):

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = xgb.XGBRegressor(**params)
        
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        preds = model.predict(X_val)
        mae_scores.append(mean_absolute_error(y_val, preds))

    return np.mean(mae_scores)

In [13]:
study = optuna.create_study(
    direction="minimize"
)

study.optimize(
    objective,
    n_trials=200,
    show_progress_bar=True
)

[I 2026-05-13 04:15:13,588] A new study created in memory with name: no-name-36b92a8f-c6d3-4af8-ac1f-b5726f259cd9


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-05-13 04:15:14,861] Trial 0 finished with value: 0.8720156788825989 and parameters: {'n_estimators': 427, 'max_depth': 4, 'learning_rate': 0.0139878616549779, 'subsample': 0.7083427323302387, 'colsample_bytree': 0.6112550037382691, 'min_child_weight': 13, 'gamma': 4.498109147360447, 'reg_lambda': 0.23280777087136076}. Best is trial 0 with value: 0.8720156788825989.
[I 2026-05-13 04:15:16,814] Trial 1 finished with value: 0.8872463583946228 and parameters: {'n_estimators': 923, 'max_depth': 4, 'learning_rate': 0.06134800112506874, 'subsample': 0.6521454543656449, 'colsample_bytree': 0.7460009618910378, 'min_child_weight': 15, 'gamma': 3.377584394382918, 'reg_lambda': 0.00017264398782990313}. Best is trial 0 with value: 0.8720156788825989.
[I 2026-05-13 04:15:18,326] Trial 2 finished with value: 0.8587258577346801 and parameters: {'n_estimators': 511, 'max_depth': 4, 'learning_rate': 0.00764884808861747, 'subsample': 0.9535210694824068, 'colsample_bytree': 0.9587410817916595, 'mi

In [14]:
trial_df = study.trials_dataframe()
fig = px.line(
    trial_df,
    x="number",
    y="value",
    title="Optuna Optimization History",template = "presentation"
)
fig.show()

In [15]:
print("Best MAE:")
print(study.best_value)

print("\nBest Parameters:")
print(study.best_params)

Best MAE:
0.8476357460021973

Best Parameters:
{'n_estimators': 800, 'max_depth': 8, 'learning_rate': 0.016938531134270917, 'subsample': 0.9809586622448973, 'colsample_bytree': 0.8102631339150652, 'min_child_weight': 14, 'gamma': 3.7355806317338365, 'reg_lambda': 9.843327067497283}


In [16]:
def evaluate(df, feature_list, best_params, target_col='G3'):
    X = df[feature_list]
    y = df[target_col]
    groups = df['student_id']
    
    gkf = GroupKFold(n_splits=5)
    
    mae_scores = []
    mse_scores = []
    r2_scores = []
    
    print(f"\n--- Training Model with {len(feature_list)} features ---")
    
    for train_idx, val_idx in gkf.split(X, y, groups=groups):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = xgb.XGBRegressor(**best_params, tree_method= "hist",objective= "reg:squarederror",random_state= 42,
              enable_categorical =True)
        
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        preds = model.predict(X_val)
        mae_scores.append(mean_absolute_error(y_val, preds))
        mse_scores.append(mean_squared_error(y_val, preds))
        r2_scores.append(r2_score(y_val, preds))
        
    print('MAE :',mae_scores)    
    print(f"Average MAE: {np.mean(mae_scores):.4f}")
    print('MSE :',mse_scores)    
    print(f"Average MSE: {np.mean(mse_scores):.4f}")
    print('r2_scores : ',r2_scores)
    print(f"Average R2: {np.mean(r2_scores):.4f}")

In [17]:
# evaluating Model 3 (post G2 exam)
evaluate(df, features_model3, best_params=study.best_params)


--- Training Model with 33 features ---
MAE : [0.7974538207054138, 0.8176916241645813, 0.8504562973976135, 0.8679565787315369, 0.9046204090118408]
Average MAE: 0.8476
MSE : [1.5725754499435425, 1.751560091972351, 1.5326460599899292, 2.4797310829162598, 2.316154956817627]
Average MSE: 1.9305
r2_scores :  [0.8852112293243408, 0.8597636222839355, 0.8977338671684265, 0.845866322517395, 0.8648068308830261]
Average R2: 0.8707


In [18]:
# final model 3 (using the whole data)
model_3 = xgb.XGBRegressor(**study.best_params, tree_method= "hist",objective= "reg:squarederror",random_state= 42,
              enable_categorical =True)
model_3.fit(
            X, y,
            verbose=True
        )
pred = model_3.predict(X)
mean_absolute_error(pred,y)

0.6771877408027649

In [19]:
model_3.save_model("model_3.json")

### Feature Importance

In [21]:
importance = model_3.feature_importances_

feature_imp_df = (
    pd.DataFrame({
        "feature": X.columns,
        "importance": importance
    })
    .sort_values(
        "importance",
        ascending=False
    )
)

feature_imp_df

,feature,importance
32,G2,0.342962
31,G1,0.284532
29,absences,0.051801
30,subject,0.020928
2,age,0.019184
14,failures,0.019112
0,school,0.016400
18,activities,0.014425
10,reason,0.012360
23,famrel,0.011956


In [22]:
X

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,subject,G1,G2
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,Maths,5,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,Maths,5,5
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,Maths,7,8
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,Maths,15,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,Maths,6,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1039,MS,F,19,R,GT3,T,2,3,services,other,...,5,4,2,1,2,5,4,Portuguese,10,11
1040,MS,F,18,U,LE3,T,3,1,teacher,services,...,4,3,4,1,1,1,4,Portuguese,15,15
1041,MS,F,18,U,GT3,T,1,1,other,other,...,1,1,1,1,1,5,6,Portuguese,11,12
1042,MS,M,17,U,LE3,T,3,1,services,services,...,2,4,5,3,4,2,6,Portuguese,10,10


In [23]:
X.columns

Index(['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
       'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime',
       'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences', 'subject', 'G1', 'G2'],
      dtype='str')

In [31]:
X.select_dtypes(include='category')

,school,sex,address,famsize,Pstatus,Mjob,Fjob,reason,guardian,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,subject
0,GP,F,U,GT3,A,at_home,teacher,course,mother,yes,no,no,no,yes,yes,no,no,Maths
1,GP,F,U,GT3,T,at_home,other,course,father,no,yes,no,no,no,yes,yes,no,Maths
2,GP,F,U,LE3,T,at_home,other,other,mother,yes,no,yes,no,yes,yes,yes,no,Maths
3,GP,F,U,GT3,T,health,services,home,mother,no,yes,yes,yes,yes,yes,yes,yes,Maths
4,GP,F,U,GT3,T,other,other,home,father,no,yes,yes,no,yes,yes,no,no,Maths
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1039,MS,F,R,GT3,T,services,other,course,mother,no,no,no,yes,no,yes,yes,no,Portuguese
1040,MS,F,U,LE3,T,teacher,services,course,mother,no,yes,no,no,yes,yes,yes,no,Portuguese
1041,MS,F,U,GT3,T,other,other,course,mother,no,no,no,yes,yes,yes,no,no,Portuguese
1042,MS,M,U,LE3,T,services,services,course,mother,no,no,no,no,no,yes,yes,no,Portuguese


In [43]:
exact_categories_data = {}
temp_df = X.select_dtypes(include='category')
for i in temp_df.columns :
    exact_categories_data[i] = list(temp_df[i].unique())


In [44]:
exact_categories_data

{'school': ['GP', 'MS'],
 'sex': ['F', 'M'],
 'address': ['U', 'R'],
 'famsize': ['GT3', 'LE3'],
 'Pstatus': ['A', 'T'],
 'Mjob': ['at_home', 'health', 'other', 'services', 'teacher'],
 'Fjob': ['teacher', 'other', 'services', 'health', 'at_home'],
 'reason': ['course', 'other', 'home', 'reputation'],
 'guardian': ['mother', 'father', 'other'],
 'schoolsup': ['yes', 'no'],
 'famsup': ['no', 'yes'],
 'paid': ['no', 'yes'],
 'activities': ['no', 'yes'],
 'nursery': ['yes', 'no'],
 'higher': ['yes', 'no'],
 'internet': ['no', 'yes'],
 'romantic': ['no', 'yes'],
 'subject': ['Maths', 'Portuguese']}

In [45]:
print(exact_categories_data)

{'school': ['GP', 'MS'], 'sex': ['F', 'M'], 'address': ['U', 'R'], 'famsize': ['GT3', 'LE3'], 'Pstatus': ['A', 'T'], 'Mjob': ['at_home', 'health', 'other', 'services', 'teacher'], 'Fjob': ['teacher', 'other', 'services', 'health', 'at_home'], 'reason': ['course', 'other', 'home', 'reputation'], 'guardian': ['mother', 'father', 'other'], 'schoolsup': ['yes', 'no'], 'famsup': ['no', 'yes'], 'paid': ['no', 'yes'], 'activities': ['no', 'yes'], 'nursery': ['yes', 'no'], 'higher': ['yes', 'no'], 'internet': ['no', 'yes'], 'romantic': ['no', 'yes'], 'subject': ['Maths', 'Portuguese']}
